# Med-Guard — Eğitim ve Değerlendirme Pipeline

Kullanıcı isteklerini **safe / unsafe** olarak sınıflandıran ve niyet (intent) tahmini yapan, işaretli graf (signed graph) tabanlı embedding propagation modeli.

Ağır/tekrar kullanılabilir mantık (config, veri işleme, graf inşası, propagation, model, kayıp fonksiyonları, eğitim ve çıkarım) `../src/` altında modüller halinde tutulur. Bu notebook onları çağırır ve sonuçları analiz eder.

## 1. Kurulum ve Kütüphaneler

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import torch
import torch.nn as nn

from sklearn.metrics import classification_report, confusion_matrix

from src.config import Config, IDX_TO_INTENT, INTENT_CLASS_WEIGHTS
from src import data, graph, propagation, evaluate
from src.model import SignedGraphSafetyModel
from src.train import train_model

## 2. Konfigürasyon

Veri yolu, embedding modeli, graf/propagation eşikleri ve eğitim hiperparametreleri.

In [ ]:
cfg = Config()

## 3. Veri Yükleme ve Embedding Çıkarımı

CSV'nin okunması, etiket/niyet haritalarının uygulanması ve çok dilli sentence-transformer ile metinlerin vektörleştirilmesi.

In [ ]:
df            = data.load_dataset(cfg.DATA_PATH)
texts         = df["request"].tolist()
labels        = df["label"].values
intent_labels = df["intent_idx"].values

print("Dataset size:", len(df))

embeddings, encoder = data.embed_texts(texts, cfg.EMBED_MODEL)
print("Embeddings shape:", embeddings.shape)

## 4. Train / Calibration / Test Ayrımı

Embedding'ler önce train/test, ardından train seti train/calibration olarak niyet etiketine göre stratifiye edilerek bölünür; tensörlere ve cihaza (GPU/CPU) taşınır.

In [ ]:
split = data.split_train_cal_test(
    embeddings, labels, intent_labels, texts,
    test_size=cfg.TEST_SIZE,
    cal_size_within_train=cfg.CAL_SIZE_WITHIN_TRAIN,
    random_state=cfg.RANDOM_STATE)

print("Train size:", len(split["X_train"]))
print("Cal size  :", len(split["X_cal"]))
print("Test size :", len(split["X_test"]))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tensors = data.to_tensors(split, device)

## 5. İşaretli Graf (Signed Graph) İnşası

Eğitim seti üzerinde k-NN benzerliğine göre kurulan graf: aynı etiketli benzer örnekler arasında **pozitif** (pull), önceden tanımlı karıştırılabilir niyet çiftlerinde zıt etiketli benzer örnekler arasında **negatif** (push) kenarlar.

In [ ]:
confusion_pairs = graph.build_confusion_pairs()

pos_i, pos_j, neg_u, neg_s, A_pos_t, A_neg_t = graph.build_signed_graph_separate(
    split["X_train"], split["y_train"], split["int_train"],
    confusion_pairs, device,
    k_graph=cfg.K_GRAPH,
    pos_sim_threshold=cfg.POS_SIM_THRESHOLD,
    neg_sim_threshold=cfg.NEG_SIM_THRESHOLD)

print("Positive edges:", len(pos_i))
print("Negative edges:", len(neg_u))

## 6. Inductive Komşuluk Hesabı

Propagation fonksiyonları `src.propagation` içinde tanımlıdır (transduktif ve inductive). Burada, görülmemiş test/calibration örnekleri için eğitim setindeki en yakın komşuluklar hesaplanır.

In [ ]:
print("Inductive neighbourhood hesaplanıyor (test)...")
test_nbr_idx, test_nbr_wts = propagation.build_inductive_neighbourhood(
    split["X_test"], split["X_train"], k=cfg.K_INDUCTIVE, sim_threshold=cfg.INDUCTIVE_SIM_THR)

print("Inductive neighbourhood hesaplanıyor (cal)...")
cal_nbr_idx, cal_nbr_wts = propagation.build_inductive_neighbourhood(
    split["X_cal"], split["X_train"], k=cfg.K_INDUCTIVE, sim_threshold=cfg.INDUCTIVE_SIM_THR)

## 7. Model Mimarisi

Paylaşılan bir gövdeden (`Linear → ReLU → Dropout`) güvenlik (binary) ve niyet (9 sınıf) başlarına dallanan ağ.

In [ ]:
model = SignedGraphSafetyModel(
    in_dim=split["X_train"].shape[1], num_intents=9).to(device)

## 8. Kayıp Fonksiyonları ve Optimizer

Ağırlıklı BCE (güvenlik), sınıf ağırlıklı CE (niyet), pull-loss, margin tabanlı push-loss ve propagation-consistency loss.

In [ ]:
intent_weights_t = torch.tensor(INTENT_CLASS_WEIGHTS, device=device)
criterion_intent = nn.CrossEntropyLoss(weight=intent_weights_t)

optimizer = torch.optim.Adam(
    model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)

## 9. Eğitim (Training Loop)

In [ ]:
model = train_model(
    model, optimizer, criterion_intent,
    tensors["X_train_t"], tensors["y_train_t"], tensors["int_train_t"],
    A_pos_t, A_neg_t, pos_i, pos_j, neg_u, neg_s,
    cfg, device)

## 10. Çıkarım (Inference) — Inductive Propagation

Eğitilmiş model ile train seti üzerinde propagation özellikleri hesaplanır; calibration ve test setleri için bu özellikler inductive komşuluk üzerinden yayılarak tahmin üretilir.

In [ ]:
print("Inference — ayrıştırılmış inductive propagation...")

results = evaluate.run_inference(
    model, tensors["X_train_t"], tensors["X_cal_t"], tensors["X_test_t"],
    A_pos_t, A_neg_t,
    cal_nbr_idx, cal_nbr_wts,
    test_nbr_idx, test_nbr_wts,
    cfg, device)

## 11. Değerlendirme — Propagation Etkisi Analizi

Lokal (propagation'sız) ve propagation sonrası tahminlerin karşılaştırılması; propagation ile kurtarılan yanlış negatiflerin (FN) incelenmesi.

In [ ]:
y_train    = split["y_train"]
y_test     = split["y_test"]
int_test   = split["int_test"]
texts_test = split["texts_test"]

probs_train_local = results["probs_train_local"]
probs_train_prop  = results["probs_train_prop"]
probs_test_local  = results["probs_test_local"]
probs_test        = results["probs_test"]

print("="*70)
print("AYRIŞTIRILMIŞ EMBEDDING PROPAGATION ANALİZİ")
print("="*70)

print(f"\nTrain seti:")
print(f"  Lokal  — unsafe ort: {probs_train_local[y_train==1].mean():.4f} | "
      f"safe ort: {probs_train_local[y_train==0].mean():.4f}")
print(f"  Prop   — unsafe ort: {probs_train_prop[y_train==1].mean():.4f} | "
      f"safe ort: {probs_train_prop[y_train==0].mean():.4f}")
print(f"  → Fark unsafe: {probs_train_prop[y_train==1].mean() - probs_train_local[y_train==1].mean():+.4f}")
print(f"  → Fark safe:   {probs_train_prop[y_train==0].mean() - probs_train_local[y_train==0].mean():+.4f}")

print(f"\nTest seti (inductive):")
print(f"  Lokal  — unsafe ort: {probs_test_local[y_test==1].mean():.4f} | "
      f"safe ort: {probs_test_local[y_test==0].mean():.4f}")
print(f"  Prop   — unsafe ort: {probs_test[y_test==1].mean():.4f} | "
      f"safe ort: {probs_test[y_test==0].mean():.4f}")
print(f"  → Fark unsafe: {probs_test[y_test==1].mean() - probs_test_local[y_test==1].mean():+.4f}")
print(f"  → Fark safe:   {probs_test[y_test==0].mean() - probs_test_local[y_test==0].mean():+.4f}")


disagree = np.abs(probs_test_local - probs_test)
recovered = ((probs_test_local < 0.5) & (probs_test > 0.5) & (y_test == 1))
print(f"\nPropagation ile kurtarılan FN: {recovered.sum()}")
if recovered.sum() > 0:
    for idx in np.where(recovered)[0]:
        print(f"  local={probs_test_local[idx]:.3f} → prop={probs_test[idx]:.3f} | "
              f"intent={IDX_TO_INTENT[int_test[idx]]} | {texts_test[idx]}")

print(f"\nOrtalama disagreement: {disagree.mean():.4f} | max: {disagree.max():.4f}")
print(f"Yüksek disagreement (>0.10) oranı: {(disagree > 0.10).mean():.4f}")

## 12. Değerlendirme — Sınıflandırma Metrikleri

Test seti üzerinde nihai (propagation sonrası) tahminlerle `classification_report` ve karışıklık matrisi.

In [ ]:
y_pred_test_binary = (probs_test >= 0.5).astype(int)
y_pred_local       = (probs_test_local >= 0.5).astype(int)

print("="*70)
print(classification_report(y_test, y_pred_test_binary,
                             target_names=["safe","unsafe"]))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_test_binary))

## 13. Yanlış Negatif (False Negative) Analizi

Modelin "unsafe" olarak tespit edemediği örneklerin detaylı listesi (en düşük olasılıktan yükseğe).

In [ ]:
pred_int_test = results["pred_int_test"]

false_negatives = []
for i, text in enumerate(texts_test):
    if y_test[i] == 1 and y_pred_test_binary[i] == 0:
        false_negatives.append({
            "text":        text,
            "local":       float(probs_test_local[i]),
            "prop":        float(probs_test[i]),
            "true_intent": IDX_TO_INTENT[int_test[i]],
            "pred_intent": IDX_TO_INTENT[pred_int_test[i]]
        })

print("="*70)
print(f"BINARY FALSE NEGATIVES | Count = {len(false_negatives)}")
print("="*70)
for x in sorted(false_negatives, key=lambda z: z["prop"])[:30]:
    print("-"*90)
    print(f"LOCAL={x['local']:.4f} → PROP={x['prop']:.4f} | "
          f"TRUE={x['true_intent']} | PRED={x['pred_intent']}")
    print("TEXT:", x["text"])